# 🧹 Data Preprocessing

This notebook cleans the raw training dataset before feature engineering.

Steps:
- Convert datetime columns
- Remove invalid passenger counts
- Remove unrealistic trip durations
- Remove GPS coordinate anomalies
- Save cleaned dataset

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [19]:
train_df = pd.read_csv("../data/raw/train.csv")

print("Original Shape:", train_df.shape)
train_df.head()

Original Shape: (1458644, 11)


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [20]:
train_df["pickup_datetime"] = pd.to_datetime(train_df["pickup_datetime"])
train_df["dropoff_datetime"] = pd.to_datetime(train_df["dropoff_datetime"])

train_df.dtypes

id                            object
vendor_id                      int64
pickup_datetime       datetime64[ns]
dropoff_datetime      datetime64[ns]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag            object
trip_duration                  int64
dtype: object

In [21]:
print("Passenger count before cleaning:")
print(train_df["passenger_count"].value_counts().sort_index())

train_df = train_df[train_df["passenger_count"] > 0]

print("Shape after removing passenger_count = 0:", train_df.shape)

Passenger count before cleaning:
passenger_count
0         60
1    1033540
2     210318
3      59896
4      28404
5      78088
6      48333
7          3
8          1
9          1
Name: count, dtype: int64
Shape after removing passenger_count = 0: (1458584, 11)


In [22]:
print("Trip duration summary before cleaning:")
train_df["trip_duration"].describe()

Trip duration summary before cleaning:


count    1.458584e+06
mean     9.594611e+02
std      5.237064e+03
min      1.000000e+00
25%      3.970000e+02
50%      6.620000e+02
75%      1.075000e+03
max      3.526282e+06
Name: trip_duration, dtype: float64

In [23]:
# Keep trips between 1 minute and 6 hours
train_df = train_df[
    (train_df["trip_duration"] >= 60) &
    (train_df["trip_duration"] <= 21600)
]

print("Shape after trip_duration cleaning:", train_df.shape)

Shape after trip_duration cleaning: (1447971, 11)


In [24]:
# NYC approximate valid coordinate range
train_df = train_df[
    (train_df["pickup_longitude"].between(-75, -73)) &
    (train_df["dropoff_longitude"].between(-75, -73)) &
    (train_df["pickup_latitude"].between(40, 42)) &
    (train_df["dropoff_latitude"].between(40, 42))
]

print("Shape after coordinate cleaning:", train_df.shape)

Shape after coordinate cleaning: (1447928, 11)


In [25]:
print("Missing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:", train_df.duplicated().sum())

Missing values:
id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

Duplicate rows: 0


In [26]:
print("Final cleaned shape:", train_df.shape)
train_df.describe()

Final cleaned shape: (1447928, 11)


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
count,1.447928e+06,1447928,1447928,1.447928e+06,1.447928e+06,1.447928e+06,1.447928e+06,1.447928e+06,1.447928e+06
mean,1.535062e+00,2016-04-01 09:57:37.466069760,2016-04-01 10:11:39.513831424,1.665375e+00,-7.397356e+01,4.075099e+01,-7.397347e+01,4.075188e+01,8.420478e+02
min,1.000000e+00,2016-01-01 00:00:17,2016-01-01 00:03:31,1.000000e+00,-7.472672e+01,4.009979e+01,-7.477543e+01,4.015374e+01,6.000000e+01
25%,1.000000e+00,2016-02-17 16:32:50.500000,2016-02-17 16:47:11.750000128,1.000000e+00,-7.399187e+01,4.073740e+01,-7.399133e+01,4.073594e+01,4.010000e+02
50%,2.000000e+00,2016-04-01 16:52:58.500000,2016-04-01 17:10:46.500000,1.000000e+00,-7.398177e+01,4.075413e+01,-7.397977e+01,4.075456e+01,6.650000e+02
75%,2.000000e+00,2016-05-15 03:36:30.249999872,2016-05-15 03:47:42.500000,2.000000e+00,-7.396742e+01,4.076836e+01,-7.396310e+01,4.076982e+01,1.076000e+03
max,2.000000e+00,2016-06-30 23:59:39,2016-07-01 00:48:20,9.000000e+00,-7.309228e+01,4.169680e+01,-7.304738e+01,4.169335e+01,2.141100e+04
std,4.987693e-01,NaN,NaN,1.314700e+00,3.795321e-02,2.798339e-02,3.581418e-02,3.228485e-02,6.623443e+02


In [27]:
train_df.to_csv("../data/processed/train_clean.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


# Conclusion

Data preprocessing is complete.

The cleaned dataset was saved as:

`data/processed/train_clean.csv`

Next step: Feature Engineering.